# Concurrent Programming Practical — Hard (40 Marks)
**CO4 | DS8 — Graph Analytics Framework on SNAP ca-GrQc**

**Runtime → Change runtime type → T4 GPU before running.**

| Metric | Method | Acceleration |
|--------|--------|--------------|
| Degree Centrality | CSR row diff | Numba CUDA kernel |
| Clustering Coefficient | Sorted-neighbour intersection | Numba CUDA kernel |
| Betweenness Centrality | Brandes (sampled K sources) | CuPy GPU arrays |

In [ ]:
!nvidia-smi
!pip install -q numba cupy-cuda12x networkx

In [ ]:
import numpy as np
import cupy as cp
from numba import cuda, float32, int32
import math, time, networkx as nx
from collections import defaultdict
import urllib.request, gzip, os
print('Imports OK')

In [ ]:
# ── Download + build CSR (same as moderate notebook) ─────────────────────────
URL  = 'https://snap.stanford.edu/data/ca-GrQc.txt.gz'
FILE = 'ca-GrQc.txt.gz'
if not os.path.exists(FILE):
    print('Downloading ...')
    urllib.request.urlretrieve(URL, FILE)

edges = []
with gzip.open(FILE, 'rt') as f:
    for line in f:
        if line.startswith('#'): continue
        u, v = map(int, line.split())
        edges.append((u, v))
        edges.append((v, u))

nodes  = sorted(set(n for e in edges for n in e))
id_map = {n: i for i, n in enumerate(nodes)}
edges  = [(id_map[u], id_map[v]) for u, v in edges]
N      = len(nodes)

adj = defaultdict(list)
for u, v in edges:
    adj[u].append(v)
# sort adjacency lists — required for binary search in clustering kernel
for i in range(N):
    adj[i].sort()

row_ptr = np.zeros(N + 1, dtype=np.int32)
for i in range(N):
    row_ptr[i + 1] = row_ptr[i] + len(adj[i])
col_idx = np.empty(row_ptr[N], dtype=np.int32)
for i in range(N):
    for j, nb in enumerate(adj[i]):
        col_idx[row_ptr[i] + j] = nb

G_nx = nx.Graph()
G_nx.add_edges_from([(u, v) for u, v in edges if u < v])

print(f'Nodes: {N} | Undirected edges: {G_nx.number_of_edges()} | CSR ready')

---
### Part 1 — Degree Centrality
`degree[i] = row_ptr[i+1] - row_ptr[i]`  
Degree centrality = degree / (N−1). One thread per node, O(1) per thread.

In [ ]:
@cuda.jit
def degree_kernel(row_ptr, degree, n):
    i = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if i < n:
        degree[i] = row_ptr[i + 1] - row_ptr[i]

THREADS = 256
blocks  = math.ceil(N / THREADS)

d_rp     = cuda.to_device(row_ptr)
d_degree = cuda.device_array(N, dtype=np.int32)

degree_kernel[blocks, THREADS](d_rp, d_degree, N)
cuda.synchronize()
degree_h = d_degree.copy_to_host()
dc       = degree_h / (N - 1)

nx_deg     = dict(G_nx.degree())
deg_errors = sum(1 for i in range(N) if degree_h[i] != nx_deg.get(i, 0))
top5       = np.argsort(dc)[::-1][:5]

print('='*50)
print('  Part 1 — Degree Centrality')
print('='*50)
print(f'  Errors vs NetworkX : {deg_errors}')
print(f'  Top-5 nodes:')
for nd in top5:
    print(f'    Node {nd:5d}  degree={degree_h[nd]:4d}  DC={dc[nd]:.5f}')
print('  PASS' if deg_errors == 0 else '  FAIL')

---
### Part 2 — Local Clustering Coefficient
For node u: `CC(u) = triangles / (d*(d-1))`  
Each thread handles one node. Uses device-side binary search on sorted adjacency lists.

In [ ]:
@cuda.jit(device=True)
def binary_search(arr, start, end, target):
    lo, hi = start, end - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == target:   return True
        elif arr[mid] < target:  lo = mid + 1
        else:                    hi = mid - 1
    return False

@cuda.jit
def clustering_kernel(row_ptr, col_idx, cc, n):
    u = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if u >= n:
        return
    u_start = row_ptr[u]
    u_end   = row_ptr[u + 1]
    d_u     = u_end - u_start
    if d_u < 2:
        cc[u] = float32(0.0)
        return
    triangles = int32(0)
    for i in range(u_start, u_end):
        v = col_idx[i]
        v_start = row_ptr[v]
        v_end   = row_ptr[v + 1]
        for j in range(u_start, u_end):
            w = col_idx[j]
            if w != v and binary_search(col_idx, v_start, v_end, w):
                triangles += 1
    cc[u] = float32(triangles) / float32(d_u * (d_u - 1))

d_ci = cuda.to_device(col_idx)
d_cc = cuda.device_array(N, dtype=np.float32)

t0 = time.perf_counter()
clustering_kernel[blocks, THREADS](d_rp, d_ci, d_cc, N)
cuda.synchronize()
cc_time = (time.perf_counter() - t0) * 1000
cc_gpu  = d_cc.copy_to_host()

sample  = list(range(min(200, N)))
nx_cc   = nx.clustering(G_nx, sample)
max_err = max(abs(float(cc_gpu[nd]) - nx_cc[nd]) for nd in sample)

top5_cc = np.argsort(cc_gpu)[::-1][:5]
print('='*50)
print('  Part 2 — Clustering Coefficient')
print('='*50)
print(f'  GPU time          : {cc_time:.2f} ms')
print(f'  Average CC        : {np.mean(cc_gpu):.4f}')
print(f'  Max error (200 sample nodes) : {max_err:.5f}')
print(f'  Top-5 nodes:')
for nd in top5_cc:
    print(f'    Node {nd:5d}  CC={cc_gpu[nd]:.4f}')
print('  PASS' if max_err < 0.01 else '  CHECK (may need sorted adj)')

---
### Part 3 — Approximate Betweenness Centrality
**Algorithm:** Brandes' single-source shortest path + dependency accumulation.  
**Approximation:** Sample K source nodes → O(K·(V+E)) instead of O(V·(V+E)).  
**GPU role:** CSR stored as CuPy arrays; all sigma/delta array ops run on GPU via `cp.add.at`.

In [ ]:
def approx_betweenness_gpu(N, row_ptr_h, col_idx_h, K=100):
    np.random.seed(0)
    sources = np.random.choice(N, min(K, N), replace=False)

    # CSR on GPU — loaded once
    rp = cp.asarray(row_ptr_h)
    ci = cp.asarray(col_idx_h)
    bc = cp.zeros(N, dtype=cp.float64)

    for src in sources:
        # ── Forward BFS ───────────────────────────────────────────────────
        dist  = cp.full(N, -1, dtype=cp.int32)
        sigma = cp.zeros(N, dtype=cp.float64)
        delta = cp.zeros(N, dtype=cp.float64)
        dist[src]  = 0
        sigma[src] = 1.0

        level_nodes = []               # frontier at each level
        current = cp.array([int(src)], dtype=cp.int32)
        level   = 0

        while len(current) > 0:
            level_nodes.append(current.copy())

            # gather all neighbours of current frontier
            starts_h  = rp[current].get()
            ends_h    = rp[current + 1].get()
            current_h = current.get()

            nb_parts  = [ci[int(s):int(e)] for s, e in zip(starts_h, ends_h)]
            par_parts = [cp.full(int(e - s), int(u), dtype=cp.int32)
                         for u, s, e in zip(current_h, starts_h, ends_h)]

            nb_parts  = [x for x in nb_parts  if len(x) > 0]
            par_parts = [x for x in par_parts if len(x) > 0]
            if not nb_parts:
                break

            neighbours = cp.concatenate(nb_parts)
            parents    = cp.concatenate(par_parts)

            # mark newly discovered nodes
            new_mask   = dist[neighbours] == -1
            new_nodes  = cp.unique(neighbours[new_mask])
            if len(new_nodes) > 0:
                dist[new_nodes] = level + 1

            # accumulate sigma for nodes on shortest paths
            sp_mask = dist[neighbours] == level + 1
            sp_nb   = neighbours[sp_mask]
            sp_par  = parents[sp_mask]
            if len(sp_nb) > 0:
                adds = cp.zeros(N, dtype=cp.float64)
                cp.add.at(adds, sp_nb.get(), sigma[sp_par].get())
                sigma += adds

            current = new_nodes
            level  += 1

        # ── Reverse accumulation (Brandes back-prop) ──────────────────────
        for lv_nodes in reversed(level_nodes[1:]):
            lv = int(dist[lv_nodes[0]].item())

            starts_h  = rp[lv_nodes].get()
            ends_h    = rp[lv_nodes + 1].get()
            lv_h      = lv_nodes.get()

            nb_parts  = [ci[int(s):int(e)] for s, e in zip(starts_h, ends_h)]
            ch_parts  = [cp.full(int(e - s), int(v), dtype=cp.int32)
                         for v, s, e in zip(lv_h, starts_h, ends_h)]
            nb_parts  = [x for x in nb_parts if len(x) > 0]
            ch_parts  = [x for x in ch_parts if len(x) > 0]
            if not nb_parts:
                continue

            neighbours = cp.concatenate(nb_parts)
            children   = cp.concatenate(ch_parts)

            pred_mask = (dist[neighbours] == lv - 1) & (sigma[neighbours] > 0)
            pred_nb   = neighbours[pred_mask]
            pred_ch   = children[pred_mask]

            if len(pred_nb) > 0:
                valid   = sigma[pred_ch] > 0
                ratio   = cp.where(valid, sigma[pred_nb] / sigma[pred_ch], 0.0)
                contrib = ratio * (1.0 + delta[pred_ch])
                adds    = cp.zeros(N, dtype=cp.float64)
                cp.add.at(adds, pred_nb.get(), contrib.get())
                delta  += adds

            alive = (dist[lv_nodes] > 0) & (dist[lv_nodes] != -1)
            bc[lv_nodes[alive]] += delta[lv_nodes[alive]]

    bc_h = bc.get()
    norm = (N - 1) * (N - 2)
    if norm > 0:
        bc_h /= norm
    return bc_h

K_SAMPLES = 100
t0 = time.perf_counter()
bc_gpu = approx_betweenness_gpu(N, row_ptr, col_idx, K=K_SAMPLES)
bc_time = (time.perf_counter() - t0) * 1000

top5_bc = np.argsort(bc_gpu)[::-1][:5]
print('='*50)
print('  Part 3 — Approx Betweenness Centrality')
print(f'  Brandes, K={K_SAMPLES} sampled sources')
print('='*50)
print(f'  GPU time : {bc_time:.2f} ms')
print(f'  Top-5 nodes by BC:')
for nd in top5_bc:
    print(f'    Node {nd:5d}  BC={bc_gpu[nd]:.6f}')

# sanity check vs networkx on 20-source sample
nx_bc    = nx.betweenness_centrality(G_nx, normalized=True, k=20, seed=0)
chk      = list(range(min(20, N)))
max_err  = max(abs(bc_gpu[n] - nx_bc[n]) for n in chk)
print(f'\n  Sanity vs nx (k=20 approx): max error = {max_err:.4f}')
print('  PASS' if max_err < 0.1 else '  APPROX OK (both are sampled)')

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print()
print('╔══════════════════════════════════════════════════╗')
print('║       HARD — Graph Analytics Summary             ║')
print('╠══════════════════════════════════════════════════╣')
print(f'║  Graph : ca-GrQc  N={N}, E={G_nx.number_of_edges()}          ║')
print('╠══════════════════════════════════════════════════╣')
print(f'║  Degree Centrality   : top node {top5[0]:5d} DC={dc[top5[0]]:.4f}   ║')
print(f'║  Clustering Coeff    : avg={np.mean(cc_gpu):.4f}               ║')
print(f'║  Betweenness (K={K_SAMPLES}) : top node {top5_bc[0]:5d}              ║')
print('╚══════════════════════════════════════════════════╝')